# 41 — Model search: the full candidate field

**Reads:** `ml.train_matrix`, `ml.test_matrix` (written by `40_train_register` §5b)
**Writes:** MLflow runs only — no tables, no registered model

Ported from `05_modelling`. This is the notebook that **earns** the hyperparameters that
notebook 40 then hardcodes.

### Why this is separate from notebook 40

Two different jobs with two different cadences:

| | `40_train_register` | `41_model_search` (this) |
|---|---|---|
| Runs | every pipeline run | rarely — when features or data change materially |
| Duration | minutes | **tens of minutes to hours** |
| Output | a registered, deployable model | the parameters, and the evidence for them |
| On stage | no | no — you show its *results* |

Splitting them means the production path is not held hostage to a grid search, and the grid
search is not something anyone runs by accident.

### Why not one notebook per model

The comparison *is* the deliverable. Every candidate below is fitted on the identical split
loaded from `ml.train_matrix` — same folds, same imputed values, same encoding. Six notebooks
would each re-split and re-impute, and the resulting numbers could not honestly be put in the
same table. MLflow separates the models; the notebook keeps them comparable.

## 1. Load the frozen split

In [ ]:
import mlflow, mlflow.sklearn
import numpy as np, pandas as pd, json, time

EXPERIMENT   = "knee-poor-outcome"
RANDOM_STATE = 42

mlflow.set_experiment(EXPERIMENT)
mlflow.autolog(disable=True)     # see MLFLOW.md — autolog buries search runs

def load(table):
    pdf = spark.table(table).toPandas()
    y = pdf.pop("outcome_label").astype(int)
    return pdf, y

X_train, y_train = load("ml.train_matrix")
X_test,  y_test  = load("ml.test_matrix")

# Column order must match what the preprocessor produced, not Delta's storage order.
X_test = X_test[X_train.columns]

BASELINE_AP = float(y_test.mean())
print(f"train {X_train.shape}  test {X_test.shape}")
print(f"prevalence / no-skill AP: {BASELINE_AP:.3f}")

## 2. Metrics — identical definitions to notebook 40

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_score, recall_score, f1_score, precision_recall_curve)

def expected_calibration_error(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.digitize(y_prob, bins[1:-1])
    ece = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            ece += (m.sum() / len(y_prob)) * abs(y_true[m].mean() - y_prob[m].mean())
    return float(ece)

def recall_at_precision(y_true, y_prob, target=0.80):
    p, r, t = precision_recall_curve(y_true, y_prob)
    ok = p[:-1] >= target
    if not ok.any():
        return 0.0, 1.0
    i = int(np.argmax(r[:-1] * ok))
    return float(r[i]), float(t[i])

def evaluate(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true)
    y_hat = (y_prob >= threshold).astype(int)
    rec80, thr80 = recall_at_precision(y_true, y_prob)
    return {
        "average_precision": float(average_precision_score(y_true, y_prob)),
        "roc_auc":           float(roc_auc_score(y_true, y_prob)),
        "brier_score":       float(brier_score_loss(y_true, y_prob)),
        "ece":               expected_calibration_error(y_true, y_prob),
        "precision_at_0.5":  float(precision_score(y_true, y_hat, zero_division=0)),
        "recall_at_0.5":     float(recall_score(y_true, y_hat, zero_division=0)),
        "f1_at_0.5":         float(f1_score(y_true, y_hat, zero_division=0)),
        "recall_at_p80":     rec80,
        "threshold_for_p80": thr80,
    }

SEARCH_TAGS = {
    "run_type":            "hyperparameter_search",
    "notebook":            "41_model_search",
    "risk_class":          "high_risk_eu_ai_act",
    "feature_set_version": "v3",
}
results = {}

## 3. Logistic regression — five variants

Straight from `05_modelling`. The point is not that any of these wins; it is that the
imbalance strategies were **tried and measured**, not assumed.

Note what SMOTE is doing here: it synthesises minority-class examples *inside the pipeline*,
so it runs per-fold and never touches the validation fold. Doing it before the split — a
common mistake — leaks synthetic neighbours of test patients into training and inflates
every metric.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

LR_VARIANTS = {
    "lr_no_weights": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ]),
    "lr_balanced": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, solver="lbfgs", class_weight="balanced",
                                     random_state=RANDOM_STATE)),
    ]),
    "lr_smote": ImbPipeline([
        ("scale", StandardScaler()),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=1000, solver="lbfgs", random_state=RANDOM_STATE)),
    ]),
    "lr_lasso_l1": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, solver="saga", penalty="l1",
                                     random_state=RANDOM_STATE)),
    ]),
    "lr_lasso_balanced": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, solver="saga", penalty="l1",
                                     class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}

with mlflow.start_run(run_name="search-logistic-regression") as parent:
    mlflow.set_tags({**SEARCH_TAGS, "model_family": "logistic_regression"})
    mlflow.log_metric("baseline_average_precision", BASELINE_AP)

    for name, est in LR_VARIANTS.items():
        with mlflow.start_run(run_name=name, nested=True):
            t0 = time.time()
            est.fit(X_train, y_train)
            m = evaluate(y_test, est.predict_proba(X_test)[:, 1])
            m["fit_seconds"] = time.time() - t0
            mlflow.set_tags({**SEARCH_TAGS, "model_family": "logistic_regression",
                             "variant": name, "interpretability": "glassbox",
                             "calibrated": "false"})
            mlflow.log_metrics(m)
            results[name] = m
            print(f"{name:20s} AP={m['average_precision']:.4f}  "
                  f"recall@P80={m['recall_at_p80']:.3f}  ({m['fit_seconds']:.0f}s)")

pd.DataFrame(results).T[["average_precision", "roc_auc", "recall_at_p80"]].round(4)

## 4. Random forest — `RandomizedSearchCV`

Your original grid, unchanged: 1,440 combinations, 50 sampled, 5-fold.

**One deliberate change: `scoring`.** The original scored on `precision`, which on an
imbalanced problem rewards a model that predicts the positive class almost never — perfect
precision on three confident patients beats useful precision on three hundred. Scoring on
`average_precision` optimises the whole precision–recall trade-off instead.

Both are run below so you can see the difference rather than take my word for it. Roughly
doubles the runtime; drop `scoring_variants` to one entry if you are short on time.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

RF_GRID = {
    "model__n_estimators":      [100, 200, 300],
    "model__max_depth":         [5, 10, 15, 20, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf":  [1, 2, 4, 8],
    "model__max_features":      ["sqrt", "log2", 0.3],
    "model__class_weight":      ["balanced", "balanced_subsample"],
}
print("combinations:", np.prod([len(v) for v in RF_GRID.values()]), "| sampling 50")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring_variants = ["average_precision", "precision"]   # trim to one if time-constrained
rf_best = {}

with mlflow.start_run(run_name="search-random-forest") as parent:
    mlflow.set_tags({**SEARCH_TAGS, "model_family": "random_forest"})
    mlflow.log_param("n_iter", 50)
    mlflow.log_param("cv_folds", 5)

    for scoring in scoring_variants:
        with mlflow.start_run(run_name=f"rf_search_{scoring}", nested=True):
            t0 = time.time()
            search = RandomizedSearchCV(
                estimator=Pipeline([("scale", StandardScaler()),
                                    ("model", RandomForestClassifier(random_state=RANDOM_STATE,
                                                                     n_jobs=-1))]),
                param_distributions=RF_GRID,
                n_iter=50, scoring=scoring, cv=cv,
                random_state=RANDOM_STATE, n_jobs=-1, verbose=1, refit=True,
            )
            search.fit(X_train, y_train)

            best = search.best_estimator_
            m = evaluate(y_test, best.predict_proba(X_test)[:, 1])
            m["cv_best_score"] = float(search.best_score_)
            m["search_seconds"] = time.time() - t0

            clean = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
            mlflow.set_tags({**SEARCH_TAGS, "model_family": "random_forest",
                             "search_scoring": scoring, "interpretability": "post_hoc_shap",
                             "calibrated": "false"})
            mlflow.log_params({f"best_{k}": v for k, v in clean.items()})
            mlflow.log_metrics(m)
            mlflow.sklearn.log_model(best, artifact_path="model")

            rf_best[scoring] = clean
            results[f"rf_{scoring}"] = m
            print(f"\nscoring={scoring}: CV {search.best_score_:.4f} | "
                  f"test AP {m['average_precision']:.4f} | recall@P80 {m['recall_at_p80']:.3f}")
            print("  best params:", json.dumps(clean))

print("\nPaste the winning params into CANDIDATES in notebook 40.")

## 5. EBM — `GridSearchCV`

Your original grid: 27 combinations over `max_rounds`, `learning_rate`, `max_leaves`,
5-fold.

**Two deliberate changes.** `scoring` moves from `roc_auc` to `average_precision`, for the
same reason as above — ROC-AUC is optimistic under imbalance. And `interactions` becomes a
searched parameter rather than fixed at 0: pairwise interactions are what let an EBM express
"a high comorbidity count matters more when the pre-op score is already low", and they stay
readable as a heat map. Setting them to 0 gives up real accuracy for interpretability you
already had.

**Slowest cell in the repo.** Consider running it as a scheduled background job rather than
interactively.

In [ ]:
from sklearn.model_selection import GridSearchCV
from interpret.glassbox import ExplainableBoostingClassifier

EBM_GRID = {
    "max_rounds":    [100, 200, 300],
    "learning_rate": [0.001, 0.01, 0.05],
    "max_leaves":    [3, 5, 7],
    "interactions":  [0, 10],          # 0 = purely additive, as in the original
}
print("combinations:", np.prod([len(v) for v in EBM_GRID.values()]))

with mlflow.start_run(run_name="search-ebm") as run:
    mlflow.set_tags({**SEARCH_TAGS, "model_family": "ebm",
                     "interpretability": "glassbox", "calibrated": "false"})
    t0 = time.time()

    search = GridSearchCV(
        estimator=ExplainableBoostingClassifier(random_state=RANDOM_STATE),
        param_grid=EBM_GRID,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
        scoring="average_precision",
        n_jobs=-1, verbose=1, return_train_score=True,
    )
    search.fit(X_train, y_train)

    best = search.best_estimator_
    m = evaluate(y_test, best.predict_proba(X_test)[:, 1])
    m["cv_best_score"] = float(search.best_score_)
    m["search_seconds"] = time.time() - t0

    mlflow.log_params({f"best_{k}": v for k, v in search.best_params_.items()})
    mlflow.log_metrics(m)
    mlflow.sklearn.log_model(best, artifact_path="model")
    results["ebm_tuned"] = m

    print(f"\nbest CV AP {search.best_score_:.4f} | test AP {m['average_precision']:.4f}")
    print("best params:", json.dumps(search.best_params_))

    # Overfitting check: a large train-CV gap means the grid found a memoriser.
    cvres = pd.DataFrame(search.cv_results_)
    row = cvres.loc[search.best_index_]
    print(f"train {row['mean_train_score']:.4f} vs CV {row['mean_test_score']:.4f} "
          f"(gap {row['mean_train_score'] - row['mean_test_score']:.4f})")

EBM_BEST_PARAMS = search.best_params_

## 6. XGBoost — the accuracy ceiling

Your parameters from `05_modelling`, unchanged.

XGBoost is here to answer the question a Level 300 audience *will* ask: *"what are you
giving up by insisting on a glassbox model?"* Quantify it. If the answer is a point or two
of average precision, that is a defensible trade for an explanation a clinician can
challenge. If it is fifteen points, you have a harder conversation — and you should be
having it in August rather than on stage.

In [ ]:
from xgboost import XGBClassifier

scale_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight = {scale_weight:.2f}")

with mlflow.start_run(run_name="xgboost") as run:
    mlflow.set_tags({**SEARCH_TAGS, "model_family": "xgboost",
                     "interpretability": "blackbox",   # queryable deployment gate
                     "calibrated": "false",
                     "purpose": "accuracy_ceiling_reference"})

    xgb = Pipeline([
        ("scale", StandardScaler()),
        ("model", XGBClassifier(
            n_estimators=300, max_depth=5, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.3, min_child_weight=10,
            reg_alpha=1.0, reg_lambda=1.0,
            scale_pos_weight=scale_weight,
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")),
    ])
    xgb.fit(X_train, y_train)

    m = evaluate(y_test, xgb.predict_proba(X_test)[:, 1])
    mlflow.log_metrics(m)
    mlflow.sklearn.log_model(xgb, artifact_path="model")
    results["xgboost"] = m
    print(f"AP {m['average_precision']:.4f} | recall@P80 {m['recall_at_p80']:.3f}")

## 7. The comparison — the slide

This table is what segment 4 shows. Sort by average precision, then read across to
`recall_at_p80`: the operating point a clinical service actually cares about.

In [ ]:
tbl = (pd.DataFrame(results).T
       [["average_precision", "roc_auc", "recall_at_p80", "threshold_for_p80",
         "brier_score", "ece"]]
       .sort_values("average_precision", ascending=False)
       .round(4))
tbl.insert(0, "lift_vs_baseline", (tbl["average_precision"] / BASELINE_AP).round(2))
display(tbl)

best_overall = tbl.index[0]
ebm_ap = results.get("ebm_tuned", {}).get("average_precision")
if ebm_ap:
    gap = tbl["average_precision"].iloc[0] - ebm_ap
    print(f"\nBest overall: {best_overall} (AP {tbl['average_precision'].iloc[0]:.4f})")
    print(f"Tuned EBM:    AP {ebm_ap:.4f}")
    print(f"Cost of insisting on a glassbox model: {gap:.4f} AP "
          f"({gap / tbl['average_precision'].iloc[0] * 100:.1f}% relative)")
    print("\nThat number is the honest answer to 'why not just use the best model?'.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Refit only what we still hold in memory; the rest are in MLflow.
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for name, est in [("EBM (tuned)", best), ("XGBoost", xgb),
                  ("LR (balanced)", LR_VARIANTS["lr_balanced"])]:
    p, r, _ = precision_recall_curve(y_test, est.predict_proba(X_test)[:, 1])
    ax.plot(r, p, lw=2, label=f"{name}  AP={average_precision_score(y_test, est.predict_proba(X_test)[:,1]):.3f}")

ax.axhline(BASELINE_AP, color="#888", ls="--", lw=1.5, label=f"no skill ({BASELINE_AP:.3f})")
ax.axhline(0.80, color="#C44E52", ls=":", lw=1.5, label="80% precision target")
ax.set_xlabel("Recall — share of poor outcomes found")
ax.set_ylabel("Precision — share of flags that are correct")
ax.set_title("Precision–recall, not ROC")
ax.legend(loc="upper right"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 8. Promote the parameters

The search does not register anything. Copy the winning parameters into `CANDIDATES` in
`40_train_register`, re-run that notebook, and let the deployment gate decide.

That indirection is deliberate: a grid search optimises one number, and a model reaching a
clinician has to satisfy four — accuracy, calibration, an operating point, and an
interpretable family. The search proposes; the gate disposes.

In [ ]:
print("Paste into CANDIDATES in 40_train_register:\n")
for scoring, params in rf_best.items():
    print(f"# random forest, searched on {scoring}")
    print(f"RandomForestClassifier(**{json.dumps(params)}, random_state=RANDOM_STATE, n_jobs=-1)\n")
print("# EBM")
print(f"ExplainableBoostingClassifier(**{json.dumps(EBM_BEST_PARAMS)}, random_state=RANDOM_STATE)")

print("\n\nAll runs for this search:")
runs = mlflow.search_runs(experiment_names=[EXPERIMENT],
                          filter_string="tags.run_type = 'hyperparameter_search'",
                          order_by=["metrics.average_precision DESC"])
cols = [c for c in ["tags.model_family", "tags.variant", "tags.search_scoring",
                    "metrics.average_precision", "metrics.recall_at_p80"] if c in runs.columns]
display(runs[cols].head(15))

---

**Next:** `42_threshold_analysis` for the operating point, `43_explainability` for the
glassbox-versus-black-box contrast.